# ✨ ClassifAI Demo - Cross-Encoder Reranking ✨

---

## Overview

This notebook demonstrates how to use the `CrossEncoderRerankerHook` to improve search result quality.

While the standard bi-encoder `VectorStore.search()` is fast, we can achieve much higher accuracy by passing retrieved results through a cross-encoder model that scores **(query, document) pairs jointly**.

This demo uses:
- **Bi-encoder model**: `sentence-transformers/all-MiniLM-L6-v2` (fast retrieval)
- **Cross-encoder reranker**: `cross-encoder/ms-marco-MiniLM-L-12-v2` (accurate ranking)

The cross-encoder runs efficiently on Apple Silicon (MPS), CUDA, or CPU.

## Setup: Import Libraries and Initialise Vectoriser

First, we'll set up our bi-encoder vectoriser for initial retrieval.

In [ ]:
import polars as pl

from classifai.indexers import VectorStore
from classifai.indexers.dataclasses import VectorStoreSearchInput
from classifai.indexers.hooks import CrossEncoderRerankerHook
from classifai.vectorisers import HuggingFaceVectoriser

## Create VectorStore Without Reranking (Baseline)

First, let's create a baseline VectorStore with just the bi-encoder scores, so we can compare results.

In [ ]:
vectoriser_hf = HuggingFaceVectoriser(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore_baseline_hf = VectorStore(
    file_name="data/fake_soc_dataset.csv",
    data_type="csv",
    vectoriser=vectoriser_hf,
    output_dir="testdata_baseline_hf",
    overwrite=True,
    quiet_mode=False,
)

## Create VectorStore With Reranking Hook

Now, let's create a VectorStore that uses the `CrossEncoderRerankerHook` to improve result ranking.

In [ ]:
tokenizer_kwargs = {"local_files_only": True}
model_kwargs = {"local_files_only": True}
rerank_hook = CrossEncoderRerankerHook()

vectorstore_reranked_hf = VectorStore(
    file_name="data/fake_soc_dataset.csv",
    data_type="csv",
    vectoriser=vectoriser_hf,
    output_dir="testdata_reranked_hf",
    overwrite=True,
    quiet_mode=False,
    hooks={
        "search_postprocess": rerank_hook,
    },
)

## Compare Results: Baseline vs. Reranked

Let's run some searches and compare how the cross-encoder reranker improves result ordering.

In [ ]:
test_query_df = pl.read_csv("data/fake_soc_eval_queries.csv")
test_query = VectorStoreSearchInput(test_query_df.rename({"label": "id", "text": "query"}).sample(1).to_pandas())

baseline_results_hf = vectorstore_baseline_hf.search(test_query, n_results=5)
print("\n" + "=" * 80)
print("BASELINE RESULTS HF (Bi-encoder only)")
print("=" * 80)
print(baseline_results_hf[["query_text", "doc_text", "score", "rank"]].to_string())

reranked_results_hf = vectorstore_reranked_hf.search(test_query, n_results=100).head(5)
print("\n" + "=" * 80)
print("RERANKED RESULTS HF (Bi-encoder + Cross-encoder Reranker)")
print("=" * 80)
print(reranked_results_hf[["query_text", "doc_text", "score", "rank"]].to_string())